# 04d - Recomendador por perfil de usuario explicable

Este notebook crea una nueva capa de recomendacion personalizada sobre el trabajo del notebook `04c_recomendador_avanzado.ipynb`.

La diferencia principal es que ya no se recomienda a partir de una sola pelicula, sino a partir de varias peliculas valoradas por un usuario. Con esas valoraciones se construye un perfil positivo, un perfil negativo opcional y recomendaciones explicables.

## 1. Idea general

Un recomendador por pelicula busca titulos parecidos a una referencia concreta. Un recomendador por perfil intenta resumir los gustos del usuario a partir de varias peliculas.

En este enfoque:
- las peliculas valoradas con nota alta construyen el perfil positivo;
- las peliculas valoradas con nota baja construyen el perfil negativo;
- las recomendaciones se ordenan por similitud al perfil positivo, penalizacion por similitud al perfil negativo, calidad y popularidad;
- cada recomendacion incluye una explicacion breve basada en generos, tags y señales de calidad.

Esto lo hace mas explicable que una caja negra: podemos ver que caracteristicas pesan en el perfil y por que una pelicula entra en la lista.

## 2. Importacion de librerias

In [1]:
from pathlib import Path
import re
import sys
import unicodedata

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

base_dir = Path.cwd().resolve()
if not (base_dir / 'src').exists():
    if (base_dir.parent / 'src').exists():
        base_dir = base_dir.parent
    else:
        for parent in base_dir.parents:
            if (parent / 'src').exists():
                base_dir = parent
                break

sys.path.insert(0, str(base_dir))

movies_path = base_dir / 'data' / 'processed' / 'movies_clean.csv'
tags_path = base_dir / 'data' / 'raw' / 'tags.csv'
translation_cache_path = base_dir / 'data' / 'processed' / 'tag_translation_cache.csv'

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 160)
pd.set_option('display.max_colwidth', 120)

## 3. Carga de datos

Se usan los mismos datos locales que en los notebooks anteriores. No se usan APIs externas, Trakt, TMDb ni deep learning.

In [2]:
required_files = {
    'movies_clean': movies_path,
    'tags': tags_path,
}

missing_files = [name for name, path in required_files.items() if not path.exists()]
if missing_files:
    raise FileNotFoundError(f'No se encontraron los archivos necesarios: {missing_files}')

movies_clean = pd.read_csv(movies_path)
tags_df = pd.read_csv(tags_path)

print('movies_clean:', movies_clean.shape)
print('tags_df:', tags_df.shape)
display(movies_clean.head())

movies_clean: (87585, 26)
tags_df: (2000072, 4)


,movieId,title,genres,year,title_clean,rating_mean,rating_count,Action,Adventure,Animation,Children,Comedy,Crime,Documentary,Drama,Fantasy,Film-Noir,Horror,IMAX,Musical,Mystery,Romance,Sci-Fi,Thriller,War,Western
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,1995.0,Toy Story,3.897438,68997,0,1,1,1,1,0,0,0,1,0,0,0,0,0,0,0,0,0,0
1,2,Jumanji (1995),Adventure|Children|Fantasy,1995.0,Jumanji,3.275758,28904,0,1,0,1,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0
2,3,Grumpier Old Men (1995),Comedy|Romance,1995.0,Grumpier Old Men,3.139447,13134,0,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance,1995.0,Waiting to Exhale,2.845331,2806,0,0,0,0,1,0,0,1,0,0,0,0,0,0,1,0,0,0,0
4,5,Father of the Bride Part II (1995),Comedy,1995.0,Father of the Bride Part II,3.059602,13154,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0


## 4. Preparacion de tags y contenido

Reutilizamos la logica de `04c`: tags normalizados, filtrados y unidos a las peliculas como `tags_combined_en`.

En este notebook no se traduce nada mediante API. Si existe `tag_translation_cache.csv`, se usa como caché local. Si un tag no aparece en la caché, se conserva su version normalizada.

In [3]:
MIN_TAG_FREQUENCY = 2
BLOCKED_TAGS = {'clv', 'grun running', 'no fa ganes'}


def normalize_text(text):
    if pd.isna(text):
        return ''

    text = str(text).strip().lower()
    text = unicodedata.normalize('NFKD', text)
    text = ''.join(character for character in text if not unicodedata.combining(character))
    text = text.replace('\u2013', '-').replace('\u2014', '-')
    text = re.sub(r'[^a-z0-9\s-]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text


def is_useful_tag(tag):
    cleaned = normalize_text(tag)
    if not cleaned or cleaned in BLOCKED_TAGS:
        return False
    if not re.search(r'[a-z]', cleaned):
        return False
    if re.fullmatch(r'[\d\s\-/]+', cleaned):
        return False

    letters = len(re.findall(r'[a-z]', cleaned))
    digits = len(re.findall(r'\d', cleaned))
    if digits > letters:
        return False
    if ' ' not in cleaned and len(cleaned) <= 3:
        return False

    return True


tags_clean = tags_df[['movieId', 'tag']].dropna(subset=['tag']).copy()
tags_clean['tag_clean'] = tags_clean['tag'].map(normalize_text)
tags_clean = tags_clean[tags_clean['tag_clean'].map(is_useful_tag)].copy()
tags_clean = tags_clean.drop_duplicates(subset=['movieId', 'tag_clean']).reset_index(drop=True)

tag_frequency = tags_clean['tag_clean'].value_counts()
tags_clean = tags_clean[tags_clean['tag_clean'].map(tag_frequency) >= MIN_TAG_FREQUENCY].copy().reset_index(drop=True)
valid_tags = set(tags_clean['tag_clean'].unique())

if translation_cache_path.exists() and translation_cache_path.stat().st_size > 0:
    translation_cache = pd.read_csv(translation_cache_path)
else:
    translation_cache = pd.DataFrame(columns=['tag_clean', 'tag_translated', 'translation_changed'])

if {'tag_clean', 'tag_translated'}.issubset(translation_cache.columns):
    translation_cache = translation_cache[['tag_clean', 'tag_translated']].copy()
    translation_cache['tag_clean'] = translation_cache['tag_clean'].map(normalize_text)
    translation_cache['tag_translated'] = translation_cache['tag_translated'].fillna('').map(normalize_text)
    translation_cache = translation_cache[
        (translation_cache['tag_clean'] != '')
        & (translation_cache['tag_translated'] != '')
        & (translation_cache['tag_clean'].isin(valid_tags))
    ].drop_duplicates(subset=['tag_clean'], keep='last')
else:
    translation_cache = pd.DataFrame(columns=['tag_clean', 'tag_translated'])

translation_map = dict(zip(translation_cache['tag_clean'], translation_cache['tag_translated']))

tags_clean['tag_en'] = tags_clean['tag_clean'].map(translation_map).fillna(tags_clean['tag_clean'])
tags_clean['tag_en'] = tags_clean['tag_en'].map(normalize_text)
tags_clean = tags_clean[tags_clean['tag_en'].map(is_useful_tag)].copy()
tags_clean = tags_clean.drop_duplicates(subset=['movieId', 'tag_en']).reset_index(drop=True)

tags_grouped_en = (
    tags_clean.groupby('movieId')['tag_en']
    .apply(lambda values: ' '.join(sorted(set(tag for tag in values if tag))))
    .reset_index(name='tags_combined_en')
)

movies_with_tags = movies_clean.merge(tags_grouped_en, on='movieId', how='left')
movies_with_tags['tags_combined_en'] = movies_with_tags['tags_combined_en'].fillna('')
movies_with_tags['genres_text'] = movies_with_tags['genres'].fillna('').str.replace('|', ' ', regex=False)
movies_with_tags['content_features'] = (
    movies_with_tags['genres_text'].str.strip() + ' ' + movies_with_tags['tags_combined_en'].str.strip()
).str.strip()

print('Tags limpios usados:', len(tags_clean))
print('Tags traducidos disponibles en cache:', len(translation_map))
display(movies_with_tags[['movieId', 'title', 'genres', 'tags_combined_en', 'content_features']].head())

Tags limpios usados: 965223
Tags traducidos disponibles en cache: 3170


,movieId,title,genres,tags_combined_en,content_features
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,2009 reissue in stereoscopic 3-d 3 dimensional 55 movies every kid should see--entertainment weekly action action fi...,Adventure Animation Children Comedy Fantasy 2009 reissue in stereoscopic 3-d 3 dimensional 55 movies every kid shoul...
1,2,Jumanji (1995),Adventure|Children|Fantasy,19th century 20th century abandoned factory actor playing multiple roles adapted from book adventure adventurer alte...,Adventure Children Fantasy 19th century 20th century abandoned factory actor playing multiple roles adapted from boo...
2,3,Grumpier Old Men (1995),Comedy|Romance,best friend duringcreditsstinger fishing funniest movies funny jack lemmon midwest minnesota old age old man old men...,Comedy Romance best friend duringcreditsstinger fishing funniest movies funny jack lemmon midwest minnesota old age ...
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance,based on novel or book characters chick flick divorce interracial relationship revenge single mother slurs,Comedy Drama Romance based on novel or book characters chick flick divorce interracial relationship revenge single m...
4,5,Father of the Bride Part II (1995),Comedy,4th wall aging baby comedy confidence contraception daughter diane keaton family fantasy father father - child relat...,Comedy 4th wall aging baby comedy confidence contraception daughter diane keaton family fantasy father father - chil...


## 5. Matriz TF-IDF y señales de calidad

La matriz TF-IDF representa generos y tags. Ademas calculamos valoracion ponderada, `rating_score` y `popularity_score`, igual que en la version avanzada por pelicula.

In [4]:
tfidf_vectorizer = TfidfVectorizer(
    lowercase=True,
    strip_accents='unicode',
    stop_words='english',
    min_df=2,
    max_df=0.85,
    token_pattern=r'(?u)\b[a-zA-Z][a-zA-Z0-9\-]{2,}\b',
)
content_matrix = tfidf_vectorizer.fit_transform(movies_with_tags['content_features'])
feature_names = tfidf_vectorizer.get_feature_names_out()


def normalize_series(series):
    numeric = pd.to_numeric(series, errors='coerce')
    min_value = numeric.min()
    max_value = numeric.max()
    if pd.isna(min_value) or pd.isna(max_value) or max_value == min_value:
        return pd.Series(0.5, index=series.index)
    return (numeric - min_value) / (max_value - min_value)


movies_advanced = movies_with_tags.copy()
movies_advanced['rating_count'] = pd.to_numeric(movies_advanced['rating_count'], errors='coerce').fillna(0)
movies_advanced['rating_mean'] = pd.to_numeric(movies_advanced['rating_mean'], errors='coerce')
movies_advanced['year'] = pd.to_numeric(movies_advanced['year'], errors='coerce').astype('Int64')

m = 20
C = movies_advanced['rating_mean'].mean()
v = movies_advanced['rating_count']
R = movies_advanced['rating_mean'].fillna(C)

movies_advanced['weighted_rating'] = ((v / (v + m)) * R) + ((m / (v + m)) * C)
movies_advanced['rating_score'] = normalize_series(movies_advanced['weighted_rating'])
movies_advanced['popularity_score'] = normalize_series(np.log1p(movies_advanced['rating_count']))

print('Matriz TF-IDF:', content_matrix.shape)
display(movies_advanced[['title', 'rating_mean', 'rating_count', 'weighted_rating', 'rating_score', 'popularity_score']].head())

Matriz TF-IDF: (87585, 24668)


,title,rating_mean,rating_count,weighted_rating,rating_score,popularity_score
0,Toy Story (1995),3.897438,68997,3.897179,0.842583,0.965346
1,Jumanji (1995),3.275758,28904,3.275571,0.660531,0.889962
2,Grumpier Old Men (1995),3.139447,13134,3.139243,0.620604,0.821625
3,Waiting to Exhale (1995),2.845331,2806,2.846462,0.534856,0.687923
4,Father of the Bride Part II (1995),3.059602,13154,3.059519,0.597255,0.821757


## 6. Busqueda flexible de peliculas

Esta funcion devuelve posibles coincidencias para ayudar a revisar que titulo se esta usando cuando el usuario escribe nombres parciales.

In [5]:
def find_movie_matches(movie_title, max_results=10):
    query = normalize_text(movie_title)
    if not query:
        return pd.DataFrame(columns=['movie_index', 'movieId', 'title', 'title_clean', 'year', 'genres', 'rating_mean', 'rating_count', 'match_type'])

    title_clean = movies_advanced.get('title_clean', movies_advanced['title']).fillna('').map(normalize_text)
    title_original = movies_advanced['title'].fillna('').map(normalize_text)

    exact_mask = (title_clean == query) | (title_original == query)
    partial_mask = title_clean.str.contains(query, regex=False) | title_original.str.contains(query, regex=False)

    matches = movies_advanced[exact_mask | partial_mask].copy()
    if matches.empty:
        return pd.DataFrame(columns=['movie_index', 'movieId', 'title', 'title_clean', 'year', 'genres', 'rating_mean', 'rating_count', 'match_type'])

    matches['movie_index'] = matches.index
    matches['match_type'] = np.where(exact_mask.loc[matches.index], 'exact', 'partial')
    matches['match_rank'] = np.where(matches['match_type'] == 'exact', 0, 1)
    columns = ['movie_index', 'movieId', 'title', 'year', 'genres', 'rating_mean', 'rating_count', 'match_type']
    if 'title_clean' in matches.columns:
        columns.insert(3, 'title_clean')

    return (
        matches.sort_values(['match_rank', 'rating_count', 'rating_mean'], ascending=[True, False, False])
        .head(max_results)[columns]
        .reset_index(drop=True)
    )


def _best_movie_match(movie_title):
    matches = find_movie_matches(movie_title, max_results=1)
    if matches.empty:
        return None
    return int(matches.loc[0, 'movie_index'])


display(find_movie_matches('The Matrix'))

,movie_index,movieId,title,title_clean,year,genres,rating_mean,rating_count,match_type
0,78699,263009,The Matrix Resurrections (2021),The Matrix Resurrections,2021,Action|Adventure|Sci-Fi,2.714523,1205,partial
1,46862,172255,The Matrix Revisited (2001),The Matrix Revisited,2001,Documentary,3.624183,153,partial
2,73281,238052,A Glitch in the Matrix (2021),A Glitch in the Matrix,2021,Documentary|Horror|Sci-Fi,3.022727,22,partial
3,28908,132490,Return to Source: The Philosophy of The Matrix (2004),Return to Source: The Philosophy of The Matrix,2004,Documentary,3.475000,20,partial
4,70661,227534,The Matrix Revolutions Revisited (2004),The Matrix Revolutions Revisited,2004,Documentary,4.750000,4,partial
5,70662,227536,The Matrix Reloaded Revisited (2004),The Matrix Reloaded Revisited,2004,Documentary,4.250000,4,partial
6,67243,216897,Sex and the Matrix (2000),Sex and the Matrix,2000,NaN,1.000000,1,partial


## 7. Construccion del perfil de usuario

El perfil positivo se calcula promediando los vectores TF-IDF de las peliculas bien valoradas, ponderados por la nota del usuario.

Si el usuario ha valorado mal algunas peliculas, se construye tambien un perfil negativo. Ese perfil se usa para penalizar peliculas demasiado parecidas a lo que el usuario no quiere.

In [6]:
def _weighted_profile_vector(movie_indices, weights):
    if not movie_indices:
        return None

    weights = np.asarray(weights, dtype=float)
    if weights.sum() == 0:
        weights = np.ones_like(weights)

    weighted_vectors = content_matrix[movie_indices].multiply(weights[:, None])
    profile = weighted_vectors.sum(axis=0) / weights.sum()
    return np.asarray(profile)


def build_user_profiles(rated_movies):
    found_rows = []
    not_found = []

    for input_title, user_rating in rated_movies.items():
        movie_index = _best_movie_match(input_title)
        if movie_index is None:
            not_found.append({'input_title': input_title, 'user_rating': user_rating})
            continue

        movie = movies_advanced.loc[movie_index]
        found_rows.append({
            'input_title': input_title,
            'matched_title': movie['title'],
            'movie_index': movie_index,
            'movieId': movie['movieId'],
            'year': movie['year'],
            'genres': movie['genres'],
            'user_rating': float(user_rating),
            'rating_count': movie['rating_count'],
            'weighted_rating': movie['weighted_rating'],
        })

    found_df = pd.DataFrame(found_rows)
    not_found_df = pd.DataFrame(not_found)

    if found_df.empty:
        raise ValueError('No se encontro ninguna pelicula del perfil indicado.')

    positive_df = found_df[found_df['user_rating'] >= 4].copy()
    negative_df = found_df[found_df['user_rating'] <= 2.5].copy()

    if positive_df.empty:
        raise ValueError('El perfil necesita al menos una pelicula positiva con rating >= 4.')

    positive_profile_vector = _weighted_profile_vector(
        positive_df['movie_index'].tolist(),
        positive_df['user_rating'].tolist(),
    )

    negative_profile_vector = None
    if not negative_df.empty:
        negative_weights = (5.5 - negative_df['user_rating']).tolist()
        negative_profile_vector = _weighted_profile_vector(negative_df['movie_index'].tolist(), negative_weights)

    return {
        'positive_profile_vector': positive_profile_vector,
        'negative_profile_vector': negative_profile_vector,
        'found_movies': found_df,
        'not_found_movies': not_found_df,
        'positive_movies': positive_df,
        'negative_movies': negative_df,
    }

## 8. Explicacion del perfil

La explicacion del perfil ayuda a responder: que generos, tags y caracteristicas parecen importar mas al usuario.

In [7]:
def _split_genres(genres_value):
    if pd.isna(genres_value):
        return []
    return [genre.strip() for genre in str(genres_value).split('|') if genre.strip()]


def _top_profile_terms(profile_vector, top_n=15):
    if profile_vector is None:
        return pd.DataFrame(columns=['feature', 'weight'])

    weights = np.asarray(profile_vector).ravel()
    positive_indices = np.where(weights > 0)[0]
    if len(positive_indices) == 0:
        return pd.DataFrame(columns=['feature', 'weight'])

    top_indices = positive_indices[np.argsort(weights[positive_indices])[::-1]][:top_n]
    return pd.DataFrame({
        'feature': feature_names[top_indices],
        'weight': weights[top_indices],
    })


def _weighted_genre_summary(profile_movies, top_n=15):
    rows = []
    for _, row in profile_movies.iterrows():
        for genre in _split_genres(row['genres']):
            rows.append({'genre': genre, 'weight': row['user_rating']})
    if not rows:
        return pd.DataFrame(columns=['genre', 'weight'])
    return (
        pd.DataFrame(rows)
        .groupby('genre', as_index=False)['weight'].sum()
        .sort_values('weight', ascending=False)
        .head(top_n)
        .reset_index(drop=True)
    )


def explain_user_profile(rated_movies, top_n=15):
    profiles = build_user_profiles(rated_movies)
    positive_movies = profiles['positive_movies']
    negative_movies = profiles['negative_movies']

    year_summary = positive_movies['year'].dropna()
    popularity_summary = positive_movies[['rating_count', 'weighted_rating']].mean(numeric_only=True)

    return {
        'found_movies': profiles['found_movies'],
        'not_found_movies': profiles['not_found_movies'],
        'top_positive_genres': _weighted_genre_summary(positive_movies, top_n=top_n),
        'top_positive_features': _top_profile_terms(profiles['positive_profile_vector'], top_n=top_n),
        'top_negative_features': _top_profile_terms(profiles['negative_profile_vector'], top_n=top_n),
        'positive_year_summary': pd.Series({
            'year_mean': year_summary.mean() if not year_summary.empty else np.nan,
            'year_min': year_summary.min() if not year_summary.empty else np.nan,
            'year_max': year_summary.max() if not year_summary.empty else np.nan,
        }),
        'positive_popularity_summary': pd.Series({
            'rating_count_mean': popularity_summary.get('rating_count', np.nan),
            'weighted_rating_mean': popularity_summary.get('weighted_rating', np.nan),
        }),
    }

## 9. Recomendador por perfil

El score final combina similitud al perfil positivo, penalizacion por perfil negativo, valoracion ponderada y popularidad.

Cada modo cambia los pesos para simular distintos estilos de recomendacion.

In [8]:
MODE_WEIGHTS = {
    'balanced': {
        'similarity': 0.55,
        'negative': -0.20,
        'rating': 0.15,
        'popularity': 0.10,
        'middle_popularity': 0.00,
        'low_popularity': 0.00,
    },
    'popular': {
        'similarity': 0.40,
        'negative': -0.15,
        'rating': 0.10,
        'popularity': 0.35,
        'middle_popularity': 0.00,
        'low_popularity': 0.00,
    },
    'under_the_radar': {
        'similarity': 0.50,
        'negative': -0.20,
        'rating': 0.25,
        'popularity': 0.00,
        'middle_popularity': 0.15,
        'low_popularity': 0.00,
    },
    'obscure': {
        'similarity': 0.60,
        'negative': -0.20,
        'rating': 0.20,
        'popularity': 0.00,
        'middle_popularity': 0.00,
        'low_popularity': 0.10,
    },
    'quality': {
        'similarity': 0.40,
        'negative': -0.15,
        'rating': 0.35,
        'popularity': 0.10,
        'middle_popularity': 0.00,
        'low_popularity': 0.00,
    },
}


def _normalize_genre_list(genres):
    if genres is None:
        return []
    if isinstance(genres, str):
        genres = [genres]
    return [str(genre).strip().lower() for genre in genres if str(genre).strip()]


def _movie_genre_set(genres_value):
    return {genre.lower() for genre in _split_genres(genres_value)}


def _apply_filters(df, min_ratings=20, min_year=None, max_year=None, include_genres=None, exclude_genres=None):
    result = df.copy()
    if min_ratings is not None:
        result = result[result['rating_count'] >= min_ratings]
    if min_year is not None:
        result = result[result['year'].fillna(-1) >= min_year]
    if max_year is not None:
        result = result[result['year'].fillna(10**9) <= max_year]

    include_genres = _normalize_genre_list(include_genres)
    exclude_genres = _normalize_genre_list(exclude_genres)
    if include_genres:
        result = result[result['genres'].apply(lambda value: set(include_genres).issubset(_movie_genre_set(value)))]
    if exclude_genres:
        result = result[~result['genres'].apply(lambda value: bool(_movie_genre_set(value) & set(exclude_genres)))]
    return result


def _popularity_label(score):
    if score >= 0.75:
        return 'popularidad alta'
    if score >= 0.40:
        return 'popularidad media'
    return 'popularidad baja'


def _top_movie_terms(movie_index, top_profile_terms, max_terms=3):
    row = content_matrix[movie_index]
    movie_terms = set(feature_names[row.nonzero()[1]])
    matches = [term for term in top_profile_terms if term in movie_terms]
    return matches[:max_terms]


def _build_explanation(row, top_profile_terms):
    matched_terms = _top_movie_terms(int(row['movie_index']), top_profile_terms, max_terms=3)
    if matched_terms:
        reason = 'Coincide con tus gustos por ' + ', '.join(matched_terms)
    else:
        reason = 'Coincide con el perfil general de contenido'

    quality = 'buena valoracion ponderada' if row['rating_score'] >= 0.65 else 'valoracion ponderada razonable'
    popularity = _popularity_label(row['popularity_score'])
    if row['negative_similarity_score'] > 0.15:
        return f'{reason}; tiene {quality} y {popularity}, aunque comparte algunos rasgos con peliculas que valoraste peor.'
    return f'{reason}; tiene {quality} y {popularity}.'


def recommend_for_user_profile(
    rated_movies,
    n=20,
    min_ratings=20,
    min_year=None,
    max_year=None,
    include_genres=None,
    exclude_genres=None,
    mode='balanced',
):
    if mode not in MODE_WEIGHTS:
        raise ValueError(f'Modo no valido. Usa uno de: {list(MODE_WEIGHTS)}')

    profiles = build_user_profiles(rated_movies)
    positive_vector = profiles['positive_profile_vector']
    negative_vector = profiles['negative_profile_vector']

    user_similarity = cosine_similarity(positive_vector, content_matrix).flatten()
    if negative_vector is None:
        negative_similarity = np.zeros(len(movies_advanced))
    else:
        negative_similarity = cosine_similarity(negative_vector, content_matrix).flatten()

    candidates = movies_advanced.copy()
    candidates['movie_index'] = candidates.index
    candidates['user_similarity_score'] = user_similarity
    candidates['negative_similarity_score'] = negative_similarity

    seen_indices = set(profiles['found_movies']['movie_index'].tolist())
    candidates = candidates[~candidates['movie_index'].isin(seen_indices)]
    candidates = _apply_filters(candidates, min_ratings=min_ratings, min_year=min_year, max_year=max_year, include_genres=include_genres, exclude_genres=exclude_genres)

    if candidates.empty:
        return pd.DataFrame(columns=[
            'title', 'year', 'genres', 'tags_combined_en', 'rating_mean', 'rating_count', 'weighted_rating',
            'rating_score', 'popularity_score', 'user_similarity_score', 'negative_similarity_score',
            'final_score', 'recommendation_mode', 'explanation'
        ])

    weights = MODE_WEIGHTS[mode]
    candidates = candidates.copy()
    candidates['middle_popularity_score'] = 1 - (candidates['popularity_score'] - 0.50).abs() / 0.50
    candidates['middle_popularity_score'] = candidates['middle_popularity_score'].clip(lower=0, upper=1)
    candidates['low_popularity_score'] = 1 - candidates['popularity_score']

    candidates['final_score'] = (
        weights['similarity'] * candidates['user_similarity_score']
        + weights['negative'] * candidates['negative_similarity_score']
        + weights['rating'] * candidates['rating_score']
        + weights['popularity'] * candidates['popularity_score']
        + weights['middle_popularity'] * candidates['middle_popularity_score']
        + weights['low_popularity'] * candidates['low_popularity_score']
    )

    top_profile_terms = explain_user_profile(rated_movies, top_n=30)['top_positive_features']['feature'].tolist()
    candidates['recommendation_mode'] = mode
    candidates['explanation'] = candidates.apply(lambda row: _build_explanation(row, top_profile_terms), axis=1)

    output_columns = [
        'title',
        'year',
        'genres',
        'tags_combined_en',
        'rating_mean',
        'rating_count',
        'weighted_rating',
        'rating_score',
        'popularity_score',
        'user_similarity_score',
        'negative_similarity_score',
        'final_score',
        'recommendation_mode',
        'explanation',
    ]

    return candidates.sort_values('final_score', ascending=False).head(n)[output_columns].reset_index(drop=True)

## 10. Ejemplo: fan de ciencia ficcion oscura

In [9]:
dark_scifi_profile = {
    'Interstellar': 5,
    'Blade Runner': 4,
    'Whiplash': 4.5,
    '28 days later': 4,
    'The great beauty': 4,
    'Gladiator': 4.5,
    'Mr. Nobody': 4,
    'Pulp fiction': 5,
}

dark_scifi_explanation = explain_user_profile(dark_scifi_profile)
display(dark_scifi_explanation['found_movies'])
display(dark_scifi_explanation['top_positive_genres'])
display(dark_scifi_explanation['top_positive_features'])
display(dark_scifi_explanation['top_negative_features'])
display(recommend_for_user_profile(dark_scifi_profile, n=10, mode='balanced'))

,input_title,matched_title,movie_index,movieId,year,genres,user_rating,rating_count,weighted_rating
0,Interstellar,Interstellar (2014),21207,109487,2014,Sci-Fi|IMAX,5.0,37157,4.132840
1,Blade Runner,Blade Runner (1982),536,541,1982,Action|Sci-Fi|Thriller,4.0,46289,4.109527
2,Whiplash,Whiplash (2014),21866,112552,2014,Drama,4.5,19477,4.153567
3,28 days later,28 Days Later (2002),6380,6502,2002,Action|Horror|Sci-Fi,4.0,21390,3.771420
4,Gladiator,Gladiator (2000),3480,3578,2000,Action|Adventure|Drama,4.5,57449,3.960850
5,Mr. Nobody,Mr. Nobody (2009),14985,79357,2009,Drama|Fantasy|Romance|Sci-Fi,4.0,4606,3.876265
6,Pulp fiction,Pulp Fiction (1994),292,296,1994,Comedy|Crime|Drama|Thriller,5.0,98409,4.196727


,genre,weight
0,Drama,18.0
1,Sci-Fi,17.0
2,Action,12.5
3,Thriller,9.0
4,Comedy,5.0
5,IMAX,5.0
6,Crime,5.0
7,Adventure,4.5
8,Fantasy,4.0
9,Horror,4.0


,feature,weight
0,oscar,0.117493
1,great,0.108007
2,best,0.106275
3,music,0.077113
4,soundtrack,0.075958
5,ending,0.070222
6,shot,0.069886
7,nominee,0.063060
8,good,0.057473
9,picture,0.053307


,feature,weight


,title,year,genres,tags_combined_en,rating_mean,rating_count,weighted_rating,rating_score,popularity_score,user_similarity_score,negative_similarity_score,final_score,recommendation_mode,explanation
0,Forrest Gump (1994),1994,Comedy|Drama|Romance|War,20th century 20th century history academy award winner academy award winning acting action adapted from book adventu...,4.052744,100296,4.052535,0.888083,0.997755,0.495678,0.0,0.505611,balanced,"Coincide con tus gustos por oscar, great, best; tiene buena valoracion ponderada y popularidad alta."
1,Star Wars: Episode IV - A New Hope (1977),1977,Action|Adventure|Sci-Fi,20th century fox 55 movies every kid should see--entertainment weekly 70mm abyss acting action action adventure acti...,4.099824,85010,4.099566,0.901857,0.983428,0.422945,0.0,0.466241,balanced,"Coincide con tus gustos por oscar, great, best; tiene buena valoracion ponderada y popularidad alta."
2,Inception (2010),2010,Action|Crime|Drama|Mystery|Sci-Fi|Thriller|IMAX,acting action adventure alternate reality ambiguous ambiguous ending architecture bathtub bd-video bechdel test fail...,4.157170,57931,4.156772,0.918611,0.950200,0.423837,0.0,0.465922,balanced,"Coincide con tus gustos por oscar, great, best; tiene buena valoracion ponderada y popularidad alta."
3,Schindler's List (1993),1993,Drama|War,afi 100 afi 100 cheers afi 100 heroes villains afi 100 movie quotes afi100 altruism amazing photography anti-semitis...,4.236990,73849,4.236657,0.942007,0.971234,0.397390,0.0,0.456989,balanced,"Coincide con tus gustos por oscar, great, best; tiene buena valoracion ponderada y popularidad alta."
4,"Lord of the Rings: The Return of the King, The (2003)",2003,Action|Adventure|Drama|Fantasy,41st century b c 5th millennium b c abyss academy award winner accusation acting action action-packed actor reprises...,4.094360,67449,4.094037,0.900238,0.963380,0.398829,0.0,0.450730,balanced,"Coincide con tus gustos por oscar, great, best; tiene buena valoracion ponderada y popularidad alta."
5,"Godfather, The (1972)",1972,Crime|Drama,100 greatest movies 20th century literature on screen academy award winner acting action actors adapted from book ad...,4.317030,66440,4.316636,0.965431,0.962074,0.379627,0.0,0.449817,balanced,"Coincide con tus gustos por oscar, great, best; tiene buena valoracion ponderada y popularidad alta."
6,"Matrix, The (1999)",1999,Action|Sci-Fi|Thriller,22nd century 555 phone number a very good moive abandoned subway station access code acid trip acting action action ...,4.156437,93808,4.156191,0.918441,0.991961,0.385217,0.0,0.448832,balanced,"Coincide con tus gustos por oscar, great, best; tiene buena valoracion ponderada y popularidad alta."
7,"Shawshank Redemption, The (1994)",1994,Crime|Drama,100 greatest movies 20th century 20th century literature on screen 5 stars abuse abuse of power accountant acting ac...,4.404614,102929,4.404342,0.991118,1.000000,0.360767,0.0,0.447090,balanced,"Coincide con tus gustos por oscar, great, best; tiene buena valoracion ponderada y popularidad alta."
8,Django Unchained (2012),2012,Action|Drama|Western,19th century 27 year old action actor playing multiple roles african american aftercreditsstinger alias ambush anima...,4.026371,32043,4.025734,0.880233,0.898895,0.409326,0.0,0.447054,balanced,"Coincide con tus gustos por oscar, great, best; tiene buena valoracion ponderada y popularidad alta."
9,Amadeus (1984),1984,Drama,18th century 70mm afi 100 ambition anamorphic blow-up austria bad acting based on a play beautiful best picture biog...,4.072641,24986,4.071787,0.893721,0.877342,0.409030,0.0,0.446759,balanced,"Coincide con tus gustos por oscar, great, best; tiene buena valoracion ponderada y popularidad alta."


## 11. Otros perfiles de ejemplo

In [10]:
animation_family_profile = {
    'Toy Story': 5,
    'Finding Nemo': 5,
    'Monsters Inc': 4.5,
    'Shrek': 4.5,
    'The Matrix': 2,
}

psychological_thriller_profile = {
    'Memento': 5,
    'Fight Club': 5,
    'Silence of the Lambs': 4.5,
    'Se7en': 4.5,
    'Dumb and Dumber': 2,
}

anti_comedy_profile = {
    'The Dark Knight': 5,
    'Inception': 5,
    'Memento': 4.5,
    'Toy Story': 2,
    'Dumb and Dumber': 2,
}

print('Animacion / familia')
display(recommend_for_user_profile(animation_family_profile, n=10, mode='balanced'))

print('Thrillers psicologicos')
display(recommend_for_user_profile(psychological_thriller_profile, n=10, mode='balanced'))

print('Penalizando comedias')
display(recommend_for_user_profile(anti_comedy_profile, n=10, mode='balanced', exclude_genres=None))

Animacion / familia


,title,year,genres,tags_combined_en,rating_mean,rating_count,weighted_rating,rating_score,popularity_score,user_similarity_score,negative_similarity_score,final_score,recommendation_mode,explanation
0,"Incredibles, The (2004)",2004,Action|Adventure|Animation|Children|Comedy,55 movies every kid should see--entertainment weekly 60s feel action adventure alter ego animated animation bd-r bd-...,3.848238,41463,3.847832,0.828131,0.921223,0.401060,0.037287,0.429468,balanced,"Coincide con tus gustos por animation, animated, oscar; tiene buena valoracion ponderada y popularidad alta."
1,Beauty and the Beast (1991),1991,Animation|Children|Fantasy|Musical|Romance|IMAX,18th century 2d animation 55 movies every kid should see--entertainment weekly 70mm affectionate alan menken almost ...,3.679188,41342,3.678862,0.778644,0.920970,0.398477,0.018233,0.424410,balanced,"Coincide con tus gustos por animation, animated, oscar; tiene buena valoracion ponderada y popularidad alta."
2,Toy Story 2 (1999),1999,Adventure|Animation|Children|Comedy|Fantasy,2009 reissue in stereoscopic 3-d abandonment ace of spades action figure action girl adoption adventure airplane air...,3.812043,32683,3.811549,0.817505,0.900608,0.387399,0.100346,0.405687,balanced,"Coincide con tus gustos por animation, oscar, kids; tiene buena valoracion ponderada y popularidad alta."
3,Toy Story 3 (2010),2010,Adventure|Animation|Children|Comedy|Fantasy|IMAX,3 dimensional 3 little aliens toy 3d sequel to 2d film 555 phone number abandoned abandonment action action figure a...,3.827643,20327,3.826835,0.821981,0.859463,0.381858,0.068910,0.405483,balanced,"Coincide con tus gustos por animation, animated, oscar; tiene buena valoracion ponderada y popularidad alta."
4,WALL·E (2008),2008,Adventure|Animation|Children|Romance|Sci-Fi,55 movies every kid should see--entertainment weekly action adventure aftercreditsstinger all ages alone amnesia and...,4.007614,39993,4.007113,0.874780,0.918096,0.358709,0.075207,0.405275,balanced,"Coincide con tus gustos por animation, animated, oscar; tiene buena valoracion ponderada y popularidad alta."
5,"Lion King, The (1994)",1994,Adventure|Animation|Children|Drama|Musical|IMAX,2d animation 55 movies every kid should see--entertainment weekly 70mm africa animacao animals animated animation an...,3.833155,51518,3.832834,0.823738,0.940036,0.344373,0.009382,0.405093,balanced,"Coincide con tus gustos por animation, animated, oscar; tiene buena valoracion ponderada y popularidad alta."
6,"Bug's Life, A (1998)",1998,Adventure|Animation|Children|Comedy,acting adventure animated animation ant-hill anthropomorphized animals buddy bugs characters childhood childhood fla...,3.558105,26736,3.557692,0.743157,0.883207,0.378392,0.030676,0.401775,balanced,"Coincide con tus gustos por animation, animated, oscar; tiene buena valoracion ponderada y popularidad alta."
7,Forrest Gump (1994),1994,Comedy|Drama|Romance|War,20th century 20th century history academy award winner academy award winning acting action adapted from book adventu...,4.052744,100296,4.052535,0.888083,0.997755,0.335719,0.088638,0.399906,balanced,"Coincide con tus gustos por oscar, great, child; tiene buena valoracion ponderada y popularidad alta."
8,Shrek 2 (2004),2004,Adventure|Animation|Children|Comedy|Musical|Romance,acceptance adult humor animation antonio banderas bad sequel based on a book body image brainless cameron diaz carto...,3.474849,26102,3.474489,0.718789,0.881128,0.386128,0.060281,0.396245,balanced,"Coincide con tus gustos por animation, oscar, kids; tiene buena valoracion ponderada y popularidad alta."
9,"Little Mermaid, The (1989)",1989,Animation|Children|Comedy|Musical|Romance,2d animation 55 movies every kid should see--entertainment weekly 70mm adapted from book alan menken almost favorite...,3.535704,17911,3.535112,0.736544,0.848500,0.367691,0.009411,0.395680,balanced,"Coincide con tus gustos por animation, animated, oscar; tiene buena valoracion ponderada y popularidad alta."


Thrillers psicologicos


,title,year,genres,tags_combined_en,rating_mean,rating_count,weighted_rating,rating_score,popularity_score,user_similarity_score,negative_similarity_score,final_score,recommendation_mode,explanation
0,Pulp Fiction (1994),1994,Comedy|Crime|Drama|Thriller,absorbing accidental killing accidental shooting achronological acting action action packed actor director actor dir...,4.196969,98409,4.196727,0.930313,0.996109,0.439274,0.110900,0.458579,balanced,"Coincide con tus gustos por ending, psychological, violence; tiene buena valoracion ponderada y popularidad alta."
1,"Usual Suspects, The (1995)",1995,Crime|Mystery|Thriller,acting action ambiguous ending anti-hero atmosphere bad acting bd-video benicio del toro boring brownface bryan sing...,4.265070,67750,4.264698,0.950220,0.963766,0.382811,0.043664,0.440723,balanced,"Coincide con tus gustos por ending, violence, best; tiene buena valoracion ponderada y popularidad alta."
2,"Shawshank Redemption, The (1994)",1994,Crime|Drama,100 greatest movies 20th century 20th century literature on screen 5 stars abuse abuse of power accountant acting ac...,4.404614,102929,4.404342,0.991118,1.000000,0.330332,0.054510,0.419448,balanced,"Coincide con tus gustos por ending, violence, best; tiene buena valoracion ponderada y popularidad alta."
3,"Clockwork Orange, A (1971)",1971,Crime|Drama|Sci-Fi|Thriller,20th century 20th century literature on screen 5 stars absurd violence absurdism actor adaptation adapted from book ...,3.985442,36818,3.984910,0.868277,0.910929,0.364961,0.046651,0.412733,balanced,"Coincide con tus gustos por ending, psychological, violence; tiene buena valoracion ponderada y popularidad alta."
4,L.A. Confidential (1997),1997,Crime|Film-Noir|Mystery|Thriller,20th century almost favorite ambition anti-hero bandage based on a book bd-video boring california call girl clearpl...,4.055649,32831,4.055009,0.888807,0.900999,0.344915,0.015403,0.410044,balanced,"Coincide con tus gustos por ending, violence, best; tiene buena valoracion ponderada y popularidad alta."
5,No Country for Old Men (2007),2007,Crime|Drama,adaptation adapted from book air duct air vent ambiguous ending amorality antelope anticlimactic assassin atmospheri...,4.050814,27384,4.050051,0.887355,0.885282,0.348628,0.030662,0.407244,balanced,"Coincide con tus gustos por ending, violence, best; tiene buena valoracion ponderada y popularidad alta."
6,Forrest Gump (1994),1994,Comedy|Drama|Romance|War,20th century 20th century history academy award winner academy award winning acting action adapted from book adventu...,4.052744,100296,4.052535,0.888083,0.997755,0.346176,0.083341,0.406716,balanced,"Coincide con tus gustos por ending, psychological, best; tiene buena valoracion ponderada y popularidad alta."
7,"Godfather, The (1972)",1972,Crime|Drama,100 greatest movies 20th century literature on screen academy award winner acting action actors adapted from book ad...,4.317030,66440,4.316636,0.965431,0.962074,0.309820,0.034841,0.404455,balanced,"Coincide con tus gustos por violence, best, murder; tiene buena valoracion ponderada y popularidad alta."
8,American Psycho (2000),2000,Crime|Horror|Mystery|Thriller,1980s music 27 year old 45 automatic absurdist humor abuse alienation alter ego ambiguous ambiguous ending ambulance...,3.685526,18868,3.684805,0.780385,0.853010,0.382219,0.064364,0.399706,balanced,"Coincide con tus gustos por ending, psychological, violence; tiene buena valoracion ponderada y popularidad alta."
9,"Departed, The (2006)",2006,Crime|Drama|Thriller,action alec baldwin americanized movie annoying soundtrack atmospheric awesome bad accents bad ending bd-video best ...,4.131371,35236,4.130732,0.910985,0.907124,0.318357,0.030897,0.396277,balanced,"Coincide con tus gustos por ending, violence, best; tiene buena valoracion ponderada y popularidad alta."


Penalizando comedias


,title,year,genres,tags_combined_en,rating_mean,rating_count,weighted_rating,rating_score,popularity_score,user_similarity_score,negative_similarity_score,final_score,recommendation_mode,explanation
0,Police Story 2013 (2013),2013,Action|Crime|Drama|Thriller,,3.149351,77,3.119605,0.614853,0.377472,0.598508,0.015203,0.456114,balanced,"Coincide con tus gustos por crime, thriller, action; tiene valoracion ponderada razonable y popularidad baja."
1,Time and Tide (Seunlau Ngaklau) (2000),2000,Action|Crime|Thriller,,3.376761,142,3.330874,0.676728,0.429989,0.567449,0.016104,0.453384,balanced,"Coincide con tus gustos por crime, thriller, action; tiene buena valoracion ponderada y popularidad media."
2,Pushpa: The Rise (2021),2021,Action|Crime|Drama|Thriller,,3.500000,20,3.252541,0.653786,0.263782,0.598508,0.015203,0.450585,balanced,"Coincide con tus gustos por crime, thriller, action; tiene buena valoracion ponderada y popularidad baja."
3,Rogue City (2020),2020,Action|Crime|Drama|Thriller,,3.259259,27,3.151099,0.624076,0.288707,0.598508,0.015203,0.448621,balanced,"Coincide con tus gustos por crime, thriller, action; tiene valoracion ponderada razonable y popularidad baja."
4,Ferry (2021),2021,Action|Crime|Drama|Thriller,,3.275000,20,3.140041,0.620838,0.263782,0.598508,0.015203,0.445643,balanced,"Coincide con tus gustos por crime, thriller, action; tiene valoracion ponderada razonable y popularidad baja."
5,Sleepless Night (Nuit blanche) (2011),2011,Action|Crime|Thriller,,3.425532,47,3.300025,0.667693,0.335407,0.567449,0.016104,0.442571,balanced,"Coincide con tus gustos por crime, thriller, action; tiene buena valoracion ponderada y popularidad baja."
6,Ratsasan (2018),2018,Action|Crime|Thriller,action crime thriller,3.600000,20,3.302541,0.668430,0.263782,0.567449,0.016104,0.435519,balanced,"Coincide con tus gustos por crime, thriller, action; tiene buena valoracion ponderada y popularidad baja."
7,Nothing to Lose (1994),1994,Action|Crime|Drama,,3.419048,525,3.403856,0.698102,0.542836,0.501883,0.017660,0.431503,balanced,"Coincide con tus gustos por crime, action, drama; tiene buena valoracion ponderada y popularidad media."
8,"Usual Suspects, The (1995)",1995,Crime|Mystery|Thriller,acting action ambiguous ending anti-hero atmosphere bad acting bd-video benicio del toro boring brownface bryan sing...,4.265070,67750,4.264698,0.950220,0.963766,0.369021,0.068118,0.428247,balanced,"Coincide con tus gustos por crime, thriller, action; tiene buena valoracion ponderada y popularidad alta."
9,Once a Thief (Zong heng si hai) (1991),1991,Action|Comedy|Crime|Thriller,,3.358824,255,3.333097,0.677379,0.480443,0.524032,0.053117,0.427245,balanced,"Coincide con tus gustos por crime, thriller, action; tiene buena valoracion ponderada y popularidad media."


## 12. Comparacion de modos

Para un mismo perfil, cambiamos el modo de recomendacion y observamos como varian los resultados.

In [11]:
mode_results = []
for mode in ['balanced', 'popular', 'under_the_radar', 'obscure', 'quality']:
    recs = recommend_for_user_profile(dark_scifi_profile, n=5, mode=mode)
    display(recs)
    compact = recs[['title', 'year', 'final_score']].copy()
    compact['mode'] = mode
    mode_results.append(compact)

mode_comparison = pd.concat(mode_results, ignore_index=True)
display(mode_comparison[['mode', 'title', 'year', 'final_score']])

,title,year,genres,tags_combined_en,rating_mean,rating_count,weighted_rating,rating_score,popularity_score,user_similarity_score,negative_similarity_score,final_score,recommendation_mode,explanation
0,Forrest Gump (1994),1994,Comedy|Drama|Romance|War,20th century 20th century history academy award winner academy award winning acting action adapted from book adventu...,4.052744,100296,4.052535,0.888083,0.997755,0.495678,0.0,0.505611,balanced,"Coincide con tus gustos por oscar, great, best; tiene buena valoracion ponderada y popularidad alta."
1,Star Wars: Episode IV - A New Hope (1977),1977,Action|Adventure|Sci-Fi,20th century fox 55 movies every kid should see--entertainment weekly 70mm abyss acting action action adventure acti...,4.099824,85010,4.099566,0.901857,0.983428,0.422945,0.0,0.466241,balanced,"Coincide con tus gustos por oscar, great, best; tiene buena valoracion ponderada y popularidad alta."
2,Inception (2010),2010,Action|Crime|Drama|Mystery|Sci-Fi|Thriller|IMAX,acting action adventure alternate reality ambiguous ambiguous ending architecture bathtub bd-video bechdel test fail...,4.157170,57931,4.156772,0.918611,0.950200,0.423837,0.0,0.465922,balanced,"Coincide con tus gustos por oscar, great, best; tiene buena valoracion ponderada y popularidad alta."
3,Schindler's List (1993),1993,Drama|War,afi 100 afi 100 cheers afi 100 heroes villains afi 100 movie quotes afi100 altruism amazing photography anti-semitis...,4.236990,73849,4.236657,0.942007,0.971234,0.397390,0.0,0.456989,balanced,"Coincide con tus gustos por oscar, great, best; tiene buena valoracion ponderada y popularidad alta."
4,"Lord of the Rings: The Return of the King, The (2003)",2003,Action|Adventure|Drama|Fantasy,41st century b c 5th millennium b c abyss academy award winner accusation acting action action-packed actor reprises...,4.094360,67449,4.094037,0.900238,0.963380,0.398829,0.0,0.450730,balanced,"Coincide con tus gustos por oscar, great, best; tiene buena valoracion ponderada y popularidad alta."


,title,year,genres,tags_combined_en,rating_mean,rating_count,weighted_rating,rating_score,popularity_score,user_similarity_score,negative_similarity_score,final_score,recommendation_mode,explanation
0,Forrest Gump (1994),1994,Comedy|Drama|Romance|War,20th century 20th century history academy award winner academy award winning acting action adapted from book adventu...,4.052744,100296,4.052535,0.888083,0.997755,0.495678,0.0,0.636294,popular,"Coincide con tus gustos por oscar, great, best; tiene buena valoracion ponderada y popularidad alta."
1,Star Wars: Episode IV - A New Hope (1977),1977,Action|Adventure|Sci-Fi,20th century fox 55 movies every kid should see--entertainment weekly 70mm abyss acting action action adventure acti...,4.099824,85010,4.099566,0.901857,0.983428,0.422945,0.0,0.603564,popular,"Coincide con tus gustos por oscar, great, best; tiene buena valoracion ponderada y popularidad alta."
2,Inception (2010),2010,Action|Crime|Drama|Mystery|Sci-Fi|Thriller|IMAX,acting action adventure alternate reality ambiguous ambiguous ending architecture bathtub bd-video bechdel test fail...,4.157170,57931,4.156772,0.918611,0.950200,0.423837,0.0,0.593966,popular,"Coincide con tus gustos por oscar, great, best; tiene buena valoracion ponderada y popularidad alta."
3,"Shawshank Redemption, The (1994)",1994,Crime|Drama,100 greatest movies 20th century 20th century literature on screen 5 stars abuse abuse of power accountant acting ac...,4.404614,102929,4.404342,0.991118,1.000000,0.360767,0.0,0.593419,popular,"Coincide con tus gustos por oscar, great, best; tiene buena valoracion ponderada y popularidad alta."
4,"Matrix, The (1999)",1999,Action|Sci-Fi|Thriller,22nd century 555 phone number a very good moive abandoned subway station access code acid trip acting action action ...,4.156437,93808,4.156191,0.918441,0.991961,0.385217,0.0,0.593117,popular,"Coincide con tus gustos por oscar, great, best; tiene buena valoracion ponderada y popularidad alta."


,title,year,genres,tags_combined_en,rating_mean,rating_count,weighted_rating,rating_score,popularity_score,user_similarity_score,negative_similarity_score,final_score,recommendation_mode,explanation
0,Marty (1955),1955,Drama|Romance,bd-r betamax butcher crying dance hall directorial debut italian american love marriage national film registry old m...,3.890203,1480,3.878401,0.837084,0.632524,0.329250,0.0,0.484138,under_the_radar,"Coincide con tus gustos por oscar, best, nominee; tiene buena valoracion ponderada y popularidad media."
1,Separate Tables (1958),1958,Drama,bd-r hotel leitmotif oscar best actor oscar best supporting actress oscar nominee best picture oscar nominee best pi...,3.598315,178,3.538392,0.737504,0.449443,0.322247,0.0,0.480332,under_the_radar,"Coincide con tus gustos por oscar, best, nominee; tiene buena valoracion ponderada y popularidad media."
2,Tender Mercies (1983),1983,Drama|Romance|Western,alcoholism bd-r betamax country music country singer guitar in netflix queue music oscar best actor oscar best scree...,3.947587,1803,3.937247,0.854318,0.649618,0.317049,0.0,0.477219,under_the_radar,"Coincide con tus gustos por oscar, best, music; tiene buena valoracion ponderada y popularidad media."
3,"Place in the Sun, A (1951)",1951,Drama|Romance,adaptation afi 100 based on a novel bd-r black and white class classic convict courtroom drama death penalty death r...,3.863938,904,3.845348,0.827403,0.589850,0.293419,0.0,0.476605,under_the_radar,"Coincide con tus gustos por oscar, best, music; tiene buena valoracion ponderada y popularidad media."
4,Born Yesterday (1950),1950,Comedy,afi 100 laughs bd-r classic congress oscar best actress oscar nominee best picture oscar nominee best picture 1950 r...,3.847030,1010,3.830681,0.823108,0.599447,0.297898,0.0,0.474892,under_the_radar,"Coincide con tus gustos por oscar, best, nominee; tiene buena valoracion ponderada y popularidad media."


,title,year,genres,tags_combined_en,rating_mean,rating_count,weighted_rating,rating_score,popularity_score,user_similarity_score,negative_similarity_score,final_score,recommendation_mode,explanation
0,Forrest Gump (1994),1994,Comedy|Drama|Romance|War,20th century 20th century history academy award winner academy award winning acting action adapted from book adventu...,4.052744,100296,4.052535,0.888083,0.997755,0.495678,0.0,0.475248,obscure,"Coincide con tus gustos por oscar, great, best; tiene buena valoracion ponderada y popularidad alta."
1,Inception (2010),2010,Action|Crime|Drama|Mystery|Sci-Fi|Thriller|IMAX,acting action adventure alternate reality ambiguous ambiguous ending architecture bathtub bd-video bechdel test fail...,4.157170,57931,4.156772,0.918611,0.950200,0.423837,0.0,0.443005,obscure,"Coincide con tus gustos por oscar, great, best; tiene buena valoracion ponderada y popularidad alta."
2,Amadeus (1984),1984,Drama,18th century 70mm afi 100 ambition anamorphic blow-up austria bad acting based on a play beautiful best picture biog...,4.072641,24986,4.071787,0.893721,0.877342,0.409030,0.0,0.436428,obscure,"Coincide con tus gustos por oscar, great, best; tiene buena valoracion ponderada y popularidad alta."
3,Star Wars: Episode IV - A New Hope (1977),1977,Action|Adventure|Sci-Fi,20th century fox 55 movies every kid should see--entertainment weekly 70mm abyss acting action action adventure acti...,4.099824,85010,4.099566,0.901857,0.983428,0.422945,0.0,0.435796,obscure,"Coincide con tus gustos por oscar, great, best; tiene buena valoracion ponderada y popularidad alta."
4,Django Unchained (2012),2012,Action|Drama|Western,19th century 27 year old action actor playing multiple roles african american aftercreditsstinger alias ambush anima...,4.026371,32043,4.025734,0.880233,0.898895,0.409326,0.0,0.431753,obscure,"Coincide con tus gustos por oscar, great, best; tiene buena valoracion ponderada y popularidad alta."


,title,year,genres,tags_combined_en,rating_mean,rating_count,weighted_rating,rating_score,popularity_score,user_similarity_score,negative_similarity_score,final_score,recommendation_mode,explanation
0,Forrest Gump (1994),1994,Comedy|Drama|Romance|War,20th century 20th century history academy award winner academy award winning acting action adapted from book adventu...,4.052744,100296,4.052535,0.888083,0.997755,0.495678,0.0,0.608876,quality,"Coincide con tus gustos por oscar, great, best; tiene buena valoracion ponderada y popularidad alta."
1,"Shawshank Redemption, The (1994)",1994,Crime|Drama,100 greatest movies 20th century 20th century literature on screen 5 stars abuse abuse of power accountant acting ac...,4.404614,102929,4.404342,0.991118,1.000000,0.360767,0.0,0.591198,quality,"Coincide con tus gustos por oscar, great, best; tiene buena valoracion ponderada y popularidad alta."
2,Inception (2010),2010,Action|Crime|Drama|Mystery|Sci-Fi|Thriller|IMAX,acting action adventure alternate reality ambiguous ambiguous ending architecture bathtub bd-video bechdel test fail...,4.157170,57931,4.156772,0.918611,0.950200,0.423837,0.0,0.586069,quality,"Coincide con tus gustos por oscar, great, best; tiene buena valoracion ponderada y popularidad alta."
3,"Godfather, The (1972)",1972,Crime|Drama,100 greatest movies 20th century literature on screen academy award winner acting action actors adapted from book ad...,4.317030,66440,4.316636,0.965431,0.962074,0.379627,0.0,0.585959,quality,"Coincide con tus gustos por oscar, great, best; tiene buena valoracion ponderada y popularidad alta."
4,Schindler's List (1993),1993,Drama|War,afi 100 afi 100 cheers afi 100 heroes villains afi 100 movie quotes afi100 altruism amazing photography anti-semitis...,4.236990,73849,4.236657,0.942007,0.971234,0.397390,0.0,0.585782,quality,"Coincide con tus gustos por oscar, great, best; tiene buena valoracion ponderada y popularidad alta."


,mode,title,year,final_score
0,balanced,Forrest Gump (1994),1994,0.505611
1,balanced,Star Wars: Episode IV - A New Hope (1977),1977,0.466241
2,balanced,Inception (2010),2010,0.465922
3,balanced,Schindler's List (1993),1993,0.456989
4,balanced,"Lord of the Rings: The Return of the King, The (2003)",2003,0.450730
5,popular,Forrest Gump (1994),1994,0.636294
6,popular,Star Wars: Episode IV - A New Hope (1977),1977,0.603564
7,popular,Inception (2010),2010,0.593966
8,popular,"Shawshank Redemption, The (1994)",1994,0.593419
9,popular,"Matrix, The (1999)",1999,0.593117


## 13. Limitaciones actuales

- No usa todavia Trakt ni historiales reales de usuario.
- Depende de las peliculas introducidas manualmente en `rated_movies`.
- Los tags vienen de usuarios y pueden contener ruido, aunque se han limpiado y filtrado.
- No usa deep learning; es un modelo vectorial basado en TF-IDF, similitud coseno y reglas de scoring.

La ventaja es que el sistema es facil de explicar: se puede justificar cada recomendacion a partir de caracteristicas concretas del perfil.